# Define Domain Boundaries and Data Product Contracts — Solution


## Domain Definitions

| Domain | Owning Team | Scope | Data Products |
|--------|------------|-------|---------------|
| Support | Support Engineering | Ticket data, agent metrics, escalation workflows, SLA tracking | `support_tickets_clean`, `agent_performance_daily`, `escalation_events` |
| Product | Product Team | User activity, feature usage, customer health scores, adoption metrics | `feature_usage_weekly`, `customer_health_scores`, `product_adoption_metrics` |
| Finance | Finance | Billing, revenue recognition, contracts, cost allocation | `revenue_summary`, `billing_records_clean`, `contract_renewals` |
| Sales | Sales | CRM accounts, pipeline, opportunities, win/loss analysis | `crm_accounts_master`, `pipeline_forecast`, `opportunity_history` |

## Data Product Contract: support_tickets_clean (Support Domain)

| Attribute | Value |
|-----------|-------|
| **Owner** | Support Engineering Lead |
| **Description** | Cleansed, deduplicated support tickets with validated fields and PII tagging |
| **Lifecycle Stage** | Published |
| **Schema** | `ticket_id` (string, PK), `created_at` (timestamp), `customer_id` (string, FK), `subject` (string), `priority` (string), `status` (string), `assigned_agent` (string), `resolution_time_hrs` (decimal), `satisfaction_score` (decimal) |
| **SLA — Freshness** | Updated every 1 hour |
| **SLA — Availability** | 99.5% query availability (Athena) |
| **Quality Checks** | (1) Uniqueness: zero duplicate `ticket_id` values. (2) Completeness: `customer_id`, `created_at`, `status` non-null on 100% of records. (3) Freshness: data no more than 2 hours old |
| **Access Control** | Read: Product, Finance, Sales teams. Column mask: none (no PII in this view — email stored separately). Row filter: none |
| **Classification** | Internal |

## Data Product Contract: customer_health_scores (Product Domain)

| Attribute | Value |
|-----------|-------|
| **Owner** | Product Engineering Lead |
| **Description** | Composite health score per customer combining usage, support sentiment, and engagement signals |
| **Lifecycle Stage** | MVP Created (pending Sales CRM integration for full scoring model) |
| **Schema** | `customer_id` (string, PK), `health_score` (decimal 0–100), `usage_score` (decimal), `support_sentiment_score` (decimal), `engagement_score` (decimal), `churn_risk` (string: low/medium/high), `scored_at` (timestamp) |
| **SLA — Freshness** | Updated every 4 hours |
| **SLA — Availability** | 99.0% (MVP target) |
| **Quality Checks** | (1) Range: `health_score` between 0 and 100. (2) Completeness: `customer_id` and `scored_at` non-null. (3) Referential integrity: every `customer_id` exists in `crm_accounts_master` |
| **Access Control** | Read: Product, Sales, Support. Column restriction: `churn_risk` restricted to Product + Sales only. Row filter: sales reps see only assigned accounts |
| **Classification** | Confidential |

## Shared Identifier: customer_id

| Attribute | Value |
|-----------|-------|
| **Name** | `customer_id` |
| **Format** | UUID v4 string (e.g., `550e8400-e29b-41d4-a716-446655440000`) |
| **Owning Domain** | Sales (source of truth from CRM system) |
| **Rules** | All domains must reference the Sales-owned `customer_id`. No domain may create its own customer identifier. When onboarding data from external systems, a mapping table (`external_id → customer_id`) is maintained by the owning domain. |
| **Consumers** | Support (joins tickets to customers), Product (joins usage to customers), Finance (joins billing to customers) |

## Platform vs. Domain Responsibility Matrix

| Responsibility | Platform Team | Domain Team |
|---------------|--------------|-------------|
| S3 bucket provisioning & lifecycle | ✅ Owns | — |
| Glue Catalog & metastore | ✅ Owns | — |
| Lake Formation policies (infrastructure) | ✅ Owns | — |
| Athena workgroups & cost controls | ✅ Owns | — |
| Pipeline templates & frameworks | ✅ Provides | Consumes |
| Quality framework (tooling) | ✅ Provides | Implements rules |
| Schema definition & evolution | — | ✅ Owns |
| Data product SLAs | — | ✅ Defines & meets |
| Data quality rule implementation | — | ✅ Owns |
| Access control decisions (who reads) | — | ✅ Decides |
| Publishing to catalog | — | ✅ Executes |
| Cross-domain identifier standards | ✅ Defines standard | ✅ Implements |
| Monitoring & alerting (infrastructure) | ✅ Owns | — |
| Monitoring & alerting (data quality) | ✅ Provides tooling | ✅ Configures & responds |